# NFL Pregame Models vs a Coin Flip: A Leakage Audit

**The surprise:** Honest NFL pregame models barely beat a coin flip, while one leaked feature looks great because it uses results from the games being scored.

**Question:** What do honest pregame validation and a deliberately leaky control reveal on the same NFL games?

**Log loss** scores predicted probabilities against actual wins: **lower is better**, and a 50/50 coin flip scores about **0.6931**.

**TL;DR**
- Honest pregame models barely beat a coin flip: the best valid past-only score is **0.6883** from a constant training home-win rate, versus **0.6931** for 50/50. Richer clean models do not improve it.
- An intentionally leaky full-season outcome feature looks far better at **0.5660**. It uses held-out results: **INVALID diagnostic, not a model**.
- On identical 2022–24 test games, teams + week improves by **0.0118** when trained with future seasons; its paired-game 95% bootstrap gap is **−0.0173 to −0.0066**, excluding zero under that resampling method.
- Reuse the paired-fold and as-of feature checks below; three observed seasons limit what the interval says about unseen seasons.

| On the same held-out games | Past-only log loss | Valid pregame use? |
|---|---:|---|
| Constant training home-win rate | **0.6883** | Yes, baseline |
| Full-season outcome feature | **0.5660** | **INVALID diagnostic, not a model** |

The paired comparison holds test games, seeds, metrics and model specifications fixed. Hindsight folds also have **more training rows**, so the gap is not a pure causal estimate of temporal leakage.


## 1. Data and scope

**Audience:** Python users testing pregame NFL win-probability models. The target is a home win in a finished, non-tied regular-season game. We use seasons 2010–25 from the attached [Kaggle scores dataset](https://www.kaggle.com/datasets/keonim/nfl-game-scores-dataset-2017-2023) and hold out each of 2022, 2023 and 2024 in turn. Each held-out season has complete later seasons available. No public-leaderboard or test-set choices are made.

The file hash pins the dataset snapshot. Ties are excluded because this is a binary audit. The code prints library versions and season coverage. Kaggle lists the dataset license as CC0.


In [ ]:
"""NFL game-outcome validation audit. This file is also embedded in the notebook."""

from collections import defaultdict, deque
from hashlib import sha256
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEEDS = (17, 43, 101)
# Each test season has at least one complete *later* season in the source.
TEST_SEASONS = (2022, 2023, 2024)
TEAM_ALIASES = {"AZ": "ARI", "JAC": "JAX", "OAK": "LV", "SD": "LAC", "STL": "LAR"}
MODELS = ("training home rate", "previous season", "teams + week", "+ prior form", "+ prior Elo")
BOOTSTRAP_SEED = 20261001
BOOTSTRAP_REPS = 4000

In [ ]:
def load_games(path):
    """Load finished regular-season games with a date and a binary home-win label."""
    raw = pd.read_csv(path)
    regular = raw.Week.str.fullmatch(r"WEEK (?:[1-9]|1[0-8])")
    games = raw.loc[
        regular & raw.GameStatus.eq("FINAL") & raw.Season.between(2010, 2025)
    ].copy()
    assert games[["HomeScore", "AwayScore", "HomeTeam", "AwayTeam", "GameDate"]].notna().all().all()
    ties = int((games.HomeScore == games.AwayScore).sum())
    games = games.loc[games.HomeScore.ne(games.AwayScore)].copy()
    games["home"] = games.HomeTeam.replace(TEAM_ALIASES)
    games["away"] = games.AwayTeam.replace(TEAM_ALIASES)
    assert games.home.ne(games.away).all()
    date_text = games.GameDate.str.replace(r"(?<=[0-9])(st|nd|rd|th)", "", regex=True)
    games["date"] = pd.to_datetime(
        date_text + " " + games.Season.astype(str), format="%B %d %Y", errors="raise"
    )
    january = games.date.dt.month.eq(1)
    games.loc[january, "date"] += pd.DateOffset(years=1)
    games["week_num"] = games.Week.str.extract(r"(\d+)").astype(int)
    games["y"] = games.HomeScore.gt(games.AwayScore).astype(int)
    games = games.sort_values(["date", "home", "away"]).reset_index(drop=True)
    assert not games.duplicated(["date", "home", "away"]).any()
    return games, raw, ties

In [ ]:
from IPython.display import display

EXPECTED_SHA256 = "4aacc374ca7a5b1264fc1dbdeba749ad22bf35f3090486d53a61f9e93fa01254"
input_root = Path('/kaggle/input')
possible = list(input_root.rglob('2010-2026_scores.csv')) if input_root.exists() else [Path('../research_data/nfl_data_gate/2010-2026_scores.csv')]
assert possible, 'The scores file was not found among the attached Kaggle inputs.'
matches = [p for p in possible if sha256(p.read_bytes()).hexdigest() == EXPECTED_SHA256]
assert matches, f'Scores files changed: {[str(p) for p in possible]}. Re-run audit and update text.'
data_path = min(matches, key=lambda p: len(p.parts))
actual_hash = EXPECTED_SHA256
games, raw, ties = load_games(data_path)
print(f'File SHA-256: {actual_hash}')
print(f'Raw rows: {len(raw):,}; usable finished, non-tied regular-season games: {len(games):,}; ties excluded: {ties}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')
display(games.groupby('Season').agg(games=('y', 'size'), home_win_rate=('y', 'mean')).round(3).tail(6))


## 2. Same held-out games, different training seasons

For each test season, **past only** trains on earlier seasons. **Past + future** trains on all other seasons through 2025, including later seasons. The test rows are identical in each pair. The second protocol is a hindsight diagnostic: future results would be unavailable at prediction time. It also has a larger training sample.


In [ ]:
def validation_folds(games, season):
    """Yield two training sets for the *same* held-out season.

    'Past only' is the deployable protocol. 'Past + future' fits on every
    other complete season through 2025, including seasons after the test.
    The latter is a hindsight diagnostic, never a valid pregame estimate.
    """
    test = np.flatnonzero(games.Season.eq(season).to_numpy())
    past = np.flatnonzero(games.Season.lt(season).to_numpy())
    hindsight = np.flatnonzero(games.Season.ne(season).to_numpy())
    future = np.flatnonzero(games.Season.gt(season).to_numpy())
    assert len(test) > 0 and len(future) > 0
    assert games.loc[past, "date"].max() < games.loc[test, "date"].min()
    assert len(np.intersect1d(hindsight, test)) == 0
    assert np.array_equal(np.union1d(past, future), hindsight)
    yield "past only", past, test
    yield "past + future", hindsight, test

## 3. Build features as of each kickoff

Team, opponent and week are known before kickoff. Recent form and Elo update only from outcomes in the training set on **earlier dates**. Unknown games cannot update feature state. The perturbation checks below test that changing future or held-out outcomes does not change clean features.


In [ ]:
def build_asof_features(games, known_indices):
    """Build form and Elo using only *training* results on earlier dates.

    All games on a date are scored from the state at the start of that date.
    Only known training results update the state afterward. This prevents a
    held-out outcomes from entering later training features.
    """
    known = np.zeros(len(games), dtype=bool)
    known[np.asarray(known_indices, dtype=int)] = True
    recent = defaultdict(lambda: deque(maxlen=5))
    elo = defaultdict(lambda: 1500.0)
    rows = np.zeros((len(games), 2), dtype=float)
    for _, group in games.groupby("date", sort=True):
        for i in group.index:
            row = games.loc[i]
            home_form = np.mean(recent[row.home]) if recent[row.home] else 0.5
            away_form = np.mean(recent[row.away]) if recent[row.away] else 0.5
            rows[i] = (home_form - away_form, (elo[row.home] - elo[row.away]) / 400.0)
        for i in group.index:
            if not known[i]:
                continue
            row = games.loc[i]
            home_win = int(row.y)
            expected = 1.0 / (1.0 + 10.0 ** ((elo[row.away] - elo[row.home]) / 400.0))
            change = 20.0 * (home_win - expected)
            elo[row.home] += change
            elo[row.away] -= change
            recent[row.home].append(home_win)
            recent[row.away].append(1 - home_win)
    return pd.DataFrame(rows, columns=["form_diff", "elo_diff"], index=games.index)


def assert_future_perturbation(games, known_indices):
    """Assert altered future outcomes cannot change earlier as-of features."""
    cutoff = pd.Timestamp("2022-09-01")
    before = build_asof_features(games, known_indices)
    changed = games.copy()
    future = changed.date.ge(cutoff)
    changed.loc[future, "y"] = 1 - changed.loc[future, "y"]
    after = build_asof_features(changed, known_indices)
    pd.testing.assert_frame_equal(before.loc[~future], after.loc[~future])
    assert future.any() and (~future).any()
    return int((~future).sum()), int(future.sum())


def assert_holdout_perturbation(games, train, test):
    """Assert validation outcomes never enter clean train or test features."""
    before = build_asof_features(games, train)
    changed = games.copy()
    changed.loc[test, "y"] = 1 - changed.loc[test, "y"]
    after = build_asof_features(changed, train)
    pd.testing.assert_frame_equal(before, after)
    return len(test)


def leaky_season_rate(games):
    """Intentionally invalid control: full-season team win rates include target games."""
    home = games[["Season", "home", "y"]].rename(columns={"home": "team", "y": "won"})
    away = games[["Season", "away", "y"]].copy()
    away["y"] = 1 - away["y"]
    away = away.rename(columns={"away": "team", "y": "won"})
    rates = pd.concat([home, away]).groupby(["Season", "team"]).won.mean()
    return pd.Series(
        [rates.loc[(row.Season, row.home)] - rates.loc[(row.Season, row.away)] for row in games.itertuples()],
        index=games.index, name="leaky_rate_diff",
    )

In [ ]:
past_2024, hindsight_2024 = list(validation_folds(games, 2024))
assert np.array_equal(past_2024[2], hindsight_2024[2])
for label, train, test in (past_2024, hindsight_2024):
    print(label, 'future perturbation:', assert_future_perturbation(games, train),
          'held-out outcomes checked:', assert_holdout_perturbation(games, train, test))

## 4. Baselines and paired experiment

**Log loss** rewards calibrated probabilities; smaller is better. A 0.5 coin flip scores about **0.693**. The constant baseline predicts the **training fold's home-win rate** for every held-out game. The previous-season baseline uses only the prior season if present in training. Logistic models add team identities and week, then prior form, then prior Elo. Each model uses identical held-out rows, seeds and scoring across the two training protocols. Preprocessing is fitted inside each fold.


In [ ]:
def previous_season_prior(games, train, test):
    """Predict the previous season's training-only home-win frequency (or 0.5)."""
    counts = games.loc[train].groupby("Season").y.mean()
    return np.array([float(counts.get(int(games.loc[i, "Season"]) - 1, 0.5)) for i in test])


def fitted_model_probability(table, y, train, test, numeric, seed):
    """Fit a new preprocessing and logistic-regression pipeline inside one fold."""
    cats = ["home", "away"]
    preprocessor = ColumnTransformer([
        ("teams", OneHotEncoder(handle_unknown="ignore"), cats),
        ("numeric", StandardScaler(), numeric),
    ])
    model = make_pipeline(preprocessor, LogisticRegression(C=1.0, max_iter=500, solver="liblinear", random_state=seed))
    model.fit(table.loc[train, cats + numeric], y.loc[train])
    return model.predict_proba(table.loc[test, cats + numeric])[:, 1]


def score_predictions(y, probability):
    """Return proper probability scores and thresholded accuracy."""
    p = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    return {
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier": brier_score_loss(y, p),
        "accuracy": accuracy_score(y, p >= 0.5),
    }


def run_audit(games, seeds=SEEDS, seasons=TEST_SEASONS):
    """Compare training histories on identical held-out games and metrics."""
    leaky = leaky_season_rate(games)
    results, diagnostics, coverage, game_losses = [], [], [], []
    for season in seasons:
        fold_data = []
        for protocol, train, test in validation_folds(games, season):
            asof = build_asof_features(games, train)
            table = games[["home", "away", "week_num"]].copy()
            table[["form_diff", "elo_diff"]] = asof
            table["leaky_rate_diff"] = leaky
            fold_data.append((protocol, train, test, table))
            coverage.append({"season": season, "protocol": protocol,
                             "train_from": games.loc[train, "date"].min().date().isoformat(),
                             "train_through": games.loc[train, "date"].max().date().isoformat(),
                             "test_from": games.loc[test, "date"].min().date().isoformat(),
                             "test_through": games.loc[test, "date"].max().date().isoformat(),
                             "n_train": len(train), "n_test": len(test),
                             "n_future_train": int(games.loc[train, "Season"].gt(season).sum())})
        # The held-out rows and their clean as-of features are literally equal.
        a, b = fold_data
        assert np.array_equal(a[2], b[2])
        pd.testing.assert_frame_equal(a[3].loc[a[2], ["form_diff", "elo_diff"]],
                                      b[3].loc[b[2], ["form_diff", "elo_diff"]])
        for seed in seeds:
            for protocol, train, test, table in fold_data:
                y_test = games.loc[test, "y"]
                predictions = {
                    "training home rate": np.full(len(test), games.loc[train, "y"].mean()),
                    "previous season": previous_season_prior(games, train, test),
                    "teams + week": fitted_model_probability(table, games.y, train, test, ["week_num"], seed),
                    "+ prior form": fitted_model_probability(table, games.y, train, test, ["week_num", "form_diff"], seed),
                    "+ prior Elo": fitted_model_probability(table, games.y, train, test, ["week_num", "form_diff", "elo_diff"], seed),
                }
                for model_name, p in predictions.items():
                    results.append({"seed": seed, "season": season, "protocol": protocol, "model": model_name,
                                    "n_train": len(train), "n_test": len(test), **score_predictions(y_test, p)})
                    clipped = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
                    labels = y_test.to_numpy()
                    point_loss = -(labels * np.log(clipped) + (1 - labels) * np.log1p(-clipped))
                    game_losses.extend({"seed": seed, "season": season, "protocol": protocol,
                                        "model": model_name, "game_index": int(i), "point_log_loss": float(loss)}
                                       for i, loss in zip(test, point_loss))
                bad = fitted_model_probability(table, games.y, train, test,
                                               ["week_num", "form_diff", "elo_diff", "leaky_rate_diff"], seed)
                diagnostics.append({"seed": seed, "season": season, "protocol": protocol,
                                    **score_predictions(y_test, bad)})
    return (pd.DataFrame(results), pd.DataFrame(diagnostics),
            pd.DataFrame(coverage), pd.DataFrame(game_losses))


def paired_game_bootstrap(game_losses, reps=BOOTSTRAP_REPS, seed=BOOTSTRAP_SEED):
    """Stratified paired game bootstrap, averaging the three held-out seasons equally.

    The paired per-game loss is averaged over fixed training seeds before
    resampling games within each season. This conditions on the trained fits
    and three observed seasons; it does not measure season-to-season uncertainty.
    """
    paired = game_losses.pivot(index=["model", "season", "game_index", "seed"],
                               columns="protocol", values="point_log_loss").reset_index()
    assert paired[["past only", "past + future"]].notna().all().all()
    paired["gap"] = paired["past + future"] - paired["past only"]
    games = paired.groupby(["model", "season", "game_index"], sort=False).gap.mean().reset_index()
    per_season = games.groupby(["model", "season"], sort=False).agg(
        gap=("gap", "mean"), n_games=("gap", "size")).reset_index()
    rng = np.random.default_rng(seed)
    intervals = []
    for model in MODELS:
        season_vectors = [g.gap.to_numpy() for _, g in games.loc[games.model.eq(model)].groupby("season")]
        draws = np.zeros(reps)
        for vector in season_vectors:
            sample_idx = rng.integers(len(vector), size=(reps, len(vector)))
            draws += vector[sample_idx].mean(axis=1) / len(season_vectors)
        low, high = np.quantile(draws, [0.025, 0.975])
        intervals.append({"model": model, "gap": np.mean([v.mean() for v in season_vectors]),
                          "ci_low": low, "ci_high": high,
                          "excludes_zero": bool(low > 0 or high < 0),
                          "bootstrap_reps": reps, "bootstrap_seed": seed})
    return per_season, pd.DataFrame(intervals)


def display_tables(results):
    """Summarize same-test scores and paired hindsight-minus-past gaps."""
    summary = (results.groupby(["protocol", "model"], sort=False)
               .agg(log_loss=("log_loss", "mean"), brier=("brier", "mean"),
                    accuracy=("accuracy", "mean"), log_loss_sd=("log_loss", "std"), folds=("log_loss", "size"))
               .reset_index())
    winners = results.loc[results.groupby(["protocol", "seed", "season"]).log_loss.idxmin()]
    wins = winners.groupby(["protocol", "model"]).size().rename("fold_wins").reset_index()
    summary = summary.merge(wins, how="left", on=["protocol", "model"]).fillna({"fold_wins": 0})
    summary["fold_wins"] = summary.fold_wins.astype(int)
    paired = (results.pivot(index=["season", "seed", "model"], columns="protocol", values="log_loss")
              .reset_index())
    paired["gap"] = paired["past + future"] - paired["past only"]
    gap = paired.groupby("model", sort=False).agg(
        past_log_loss=("past only", "mean"), hindsight_log_loss=("past + future", "mean"),
        gap=("gap", "mean"), gap_sd=("gap", "std"), paired_tests=("gap", "size")
    ).reset_index()
    gap = gap.set_index("model").loc[list(MODELS)].reset_index()
    return summary, gap, paired

In [ ]:
from IPython.display import display
start = time.perf_counter()
results, diagnostic, coverage, game_losses = run_audit(games)
summary, gap, paired = display_tables(results)
per_season, intervals = paired_game_bootstrap(game_losses)
assert np.allclose(gap.gap, intervals.set_index('model').loc[gap.model, 'gap'])
elapsed = time.perf_counter() - start
print(f'Experiment runtime: {elapsed:.1f} s')
try:
    import resource
    print('Peak RSS (platform units):', resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
except ImportError:
    pass
output_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
for name, table in [('nfl_fold_results.csv', results), ('nfl_paired_gap.csv', gap),
                    ('nfl_paired_folds.csv', paired), ('nfl_leakage_diagnostic.csv', diagnostic),
                    ('nfl_date_coverage.csv', coverage), ('nfl_per_season_gap.csv', per_season),
                    ('nfl_paired_game_bootstrap.csv', intervals), ('nfl_game_losses.csv', game_losses)]:
    table.to_csv(output_root / name, index=False)
print('Paired log-loss gap = past + future minus past only; negative favors hindsight.')
display(intervals.round(4))

### First result: honest baseline versus impossible diagnostic

Both bars score the same held-out games with past-only model fitting. The red bar still sees held-out outcomes through its full-season feature: **INVALID diagnostic, not a model**.


In [ ]:
import matplotlib.pyplot as plt
honest = float(summary.query('protocol == "past only" and model == "training home rate"').log_loss.iloc[0])
invalid = float(diagnostic.query('protocol == "past only"').log_loss.mean())
fig, ax = plt.subplots(figsize=(9.5, 4.2))
bars = ax.bar(['Honest training home rate\n(valid baseline)',
               'Leaked full-season rate\nINVALID diagnostic, not a model'],
              [honest, invalid], color=['#17699b', '#b94c49'], width=.55)
ax.axhline(np.log(2), color='0.25', linestyle='--', label='50/50 coin flip: 0.6931')
for bar, score in zip(bars, [honest, invalid]):
    ax.text(bar.get_x() + bar.get_width()/2, score + .006, f'{score:.4f}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, .78)
ax.set_ylabel('Past-only held-out log loss (lower is better)')
ax.set_title('A leaky feature looks much better on the same games')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

**What this shows:** The honest baseline barely improves on guessing, while the leaked feature looks better because it includes answers from the games being scored.


### Paired result: future-season training on the same held-out games

Each gray dot is one held-out season averaged over three seeds; the blue dot and whiskers are the mean and a 95% **paired-game bootstrap interval**. For each of 4,000 draws, we resample games with replacement *within each season* and keep the past-only and hindsight losses paired for that game. We average the three season means equally, using random seed 20261001. A negative gap favors hindsight training. The interval conditions on these three seasons and fitted models; games involving shared teams are not fully independent, so it may be too narrow for generalizing to another season.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(MODELS):
    values = per_season.loc[per_season.model.eq(name), 'gap']
    ax.scatter(values, np.full(len(values), i), color='0.60', s=34,
               label='Held-out season' if i == 0 else None)
    row = intervals.set_index('model').loc[name]
    mean_gap = row.gap
    ax.errorbar(mean_gap, i, xerr=[[mean_gap - row.ci_low], [row.ci_high - mean_gap]],
                fmt='o', color='#1565a3', markersize=8, capsize=4,
                label='Mean and 95% paired-game interval' if i == 0 else None)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(range(len(MODELS)), MODELS)
ax.invert_yaxis()
ax.set_xlabel('Past + future minus past-only log loss (lower is better)')
ax.set_title('Same held-out games: hindsight minus past-only log loss')
ax.grid(axis='x', alpha=.2)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
fig.tight_layout()
plt.show()

**What this shows:** Training with later seasons lowered log loss on these same games, but those later results would not have been available for real pregame predictions.


**Interpretation.** The biggest paired change is teams + week: **−0.0118** (0.6981 to 0.6863). Its paired-game 95% interval is **−0.0173 to −0.0066**, so **yes: under this game-level bootstrap, the gap is distinguishable from zero**. Prior form and Elo intervals also exclude zero; the baseline and previous-season intervals do not. This does **not** establish a reliable gain for unseen seasons: we observed only three test seasons, games are related, and hindsight has more training rows. In the actual past-only setting, every richer clean model loses to the home-rate baseline (**0.6883**), only a small improvement over a coin flip (**0.6931**).


In [ ]:
print('Per-season paired log loss (averaged over fixed training seeds):')
season_scores = paired.groupby(['model', 'season'], as_index=False)[['past only', 'past + future', 'gap']].mean()
display(season_scores.round(4))
print('Paired-game bootstrap intervals:')
display(intervals.round(4))
print('Full clean scores (same held-out games):')
display(summary.round(4))
print('Training and test coverage:')
display(coverage)
print('Seed variation: all three fixed seeds give identical scores for each model and season.')

## 5. Deliberately leaky diagnostic

This **INVALID diagnostic, not a model** computes each team's full-season win rate using outcomes from that season, including the held-out games it predicts. It shows how an impossible outcome feature can make a score look strong. Its score is not achievable pregame performance.


In [ ]:
diagnostic_mean = diagnostic.groupby('protocol')[['log_loss', 'brier', 'accuracy']].mean()
display(diagnostic_mean.round(4))
clean_elo_past = float(summary.query('protocol == "past only" and model == "+ prior Elo"').log_loss.iloc[0])
print(f'Invalid apparent log-loss gain versus clean past-only + prior Elo: '
      f'{clean_elo_past - diagnostic_mean.loc["past only", "log_loss"]:.4f}')

## 6. How to reuse this audit

> Swap `TEST_SEASONS` for your held-out years and replace `build_asof_features()` with your own pregame feature builder. Keep `validation_folds()` and the future/held-out outcome perturbation assertions, and refit preprocessing inside **each** fold. Check that each paired comparison scores the same game IDs.


## 7. Conclusion

- **Measured:** The best valid past-only log loss was **0.6883** versus **0.6931** for a coin flip; the **0.5660** leaky result is an **INVALID diagnostic, not a model**. Teams + week had a **−0.0118** hindsight gap on identical games, with a paired-game 95% interval of **[−0.0173, −0.0066]** under within-season resampling.
- **Practical recommendation:** For real pregame work, validate on later seasons using only earlier training results. Start with the constant training home-rate baseline and require a richer clean model to beat it before claiming an improvement.
- **Limits:** Only three test seasons were observed; games share teams, the hindsight fits have more training rows, and the game bootstrap cannot measure uncertainty across future seasons. Dates lack exact result-availability times, so the as-of features use earlier calendar dates only.
